# Rex Bot – FinRED (+ mẫu vô hại) → typed-decisions

**FinRED** (`datumo/FinRED`, KDD 2026): 5.805 prompt tấn công chatbot tài chính, 26 loại đe dọa (R1–R5).
FinRED **chỉ có mẫu độc hại**, nên notebook ghép thêm 2 nguồn vô hại cùng chủ đề để model không học
"cứ hỏi về tài chính / bảo mật là chặn":

| Nguồn | Vai trò | Nhãn |
|---|---|---|
| `datumo/FinRED` | Tấn công chatbot ngân hàng (vượt 2FA, rửa tiền, lẩn tránh tuân thủ...) | Độc hại, mức theo nhóm R1–R5 |
| `gbharti/finance-alpaca` | Câu hỏi tài chính thường ngày | Vô hại |
| `sujet-ai/Sujet-Finance-Instruct-177k` (cột `user_prompt`) | Yêu cầu tài chính **dài** (ngữ cảnh + câu hỏi), lấy mẫu khớp độ dài FinRED | Vô hại |
| `Trendyol/Trendyol-Cybersecurity-Instruction-Tuning-Dataset` (cột `user`) | Câu hỏi bảo mật "nghe đáng sợ nhưng hợp lệ" | Vô hại / mơ hồ (nhãn yếu từ việc bị từ chối) |

Cùng **5 câu hỏi và format** với bộ BAF và bộ WildGuardMix + SQLi → gộp trực tiếp được.

### Chuẩn bị (1 lần) – chọn **một** trong hai cách

**Cách 1 – Kaggle Input (không cần Internet):** tải file từ 3 trang Hugging Face trên
(tab *Files and versions*), tạo **Datasets → + New Dataset** (để **Private**), rồi **Add Input**.
FinRED: tải **tất cả** file `R*_prompts.csv` (vd `R1_1_prompts.csv` … `R5_7_prompts.csv`) và upload chung một dataset; notebook tự nhận theo mẫu tên file, mã loại đe dọa lấy từ tên file. Ba nguồn còn lại: tên thư mục/file chứa `finance-alpaca` (hoặc `finance_alpaca`), `sujet` và `trendyol`.

**Cách 2 – Tự tải:** **Settings → Internet: On**. Notebook tự tải nguồn nào chưa có trong Input.

Sau đó: **Accelerator: None → Save Version → Save & Run All**.

## 1. Cấu hình

In [ ]:
# ====== CHỈNH Ở ĐÂY ======
OUT_DIR        = "/kaggle/working/rexbot_finred"
INPUT_DIR      = "/kaggle/input"

FINANCE_MAX    = 5_000     # số câu tài chính vô hại NGẮN (finance-alpaca)
SUJET_MAX      = 10_000    # số câu tài chính vô hại DÀI (Sujet-Finance), khớp độ dài FinRED -> phá đường tắt độ dài
TRENDYOL_MAX   = 6_000     # số câu bảo mật (Trendyol) sau khi gộp câu gần-trùng
FINRED_TEST_PCT  = 0.25    # FinRED chỉ có 1 split -> tách 25% test, 10% valid (theo từng nhóm R)
# Giữ lại HOÀN TOÀN 1 loại đe dọa mỗi nhóm R -> test_unseen ("loại tấn công chưa từng thấy").
#   "auto" = tự chọn ngẫu nhiên theo SEED; hoặc ghi rõ, vd ["R1_6", "R2_5", "R3_3", "R4_5", "R5_7"]
HOLDOUT_SUBCATS  = "auto"
FINRED_VALID_PCT = 0.10
VALID_PCT      = 0.10      # 2 nguồn vô hại: 80/10/10
TEST_PCT       = 0.10

# mức độc hại theo nhóm đe dọa của FinRED
#   R1 tấn công mạng, R2 tội phạm tài chính  -> độc hại rõ
#   R3 thông tin sai lệch (tin giả, thao túng) -> độc hại rõ, phần lớn BLOCK
#   R4 vi phạm quyền người tiêu dùng, R5 lẩn tránh tuân thủ -> tác hại gián tiếp, nghiêng ESCALATE
FINRED_RISK    = {"R1": 1.0, "R2": 1.0, "R3": 0.9, "R4": 0.7, "R5": 0.7}
FINRED_IMPACT  = {"R4", "R5"}

HEAD, TAIL     = 700, 400
N_ANNOTATORS   = 30
NOISE          = 0.3
PROTECT_LEGIT  = True
FINRED_NEED_SCALE = 0.3   # prompt FinRED là yêu cầu độc hại TƯỜNG MINH -> hiếm khi cần điều tra thêm
LENGTH_MATCH   = True      # lấy mẫu câu tài chính vô hại khớp phân bố độ dài của FinRED (chống đường tắt độ dài)
SEED           = 42

In [ ]:
import hashlib, json, os, re, time, unicodedata, zipfile, glob, zlib
import numpy as np
import pandas as pd

def log(msg): print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)
def jdump(o): return json.dumps(o, ensure_ascii=False, separators=(",", ":"))

INPUT_DIR = os.environ.get("REXBOT_INPUT", INPUT_DIR)
os.makedirs(OUT_DIR, exist_ok=True)

DATA_EXT = (".parquet", ".jsonl", ".json", ".csv")

def read_any(path):
    p = path.lower()
    if p.endswith(".parquet"): return pd.read_parquet(path)
    if p.endswith(".jsonl"):   return pd.read_json(path, lines=True)
    if p.endswith(".json"):
        try:    return pd.read_json(path)
        except ValueError: return pd.read_json(path, lines=True)
    return pd.read_csv(path, encoding="utf-8", encoding_errors="ignore")

def find_files(*keywords):
    return sorted(f for f in glob.glob(f"{INPUT_DIR}/**/*", recursive=True)
                  if os.path.isfile(f) and f.lower().endswith(DATA_EXT)
                  and any(k in f.lower() for k in keywords))

def load_source(name, keywords, hf_repo, hf_split):
    """Ưu tiên file trong Kaggle Input; nếu không có thì tải từ Hugging Face (cần Internet)."""
    files = find_files(*keywords)
    if files:
        log(f"[{name}] đọc từ Input: {[os.path.basename(f) for f in files]}")
        return pd.concat([read_any(f) for f in files], ignore_index=True)
    try:
        from datasets import load_dataset
        log(f"[{name}] không có trong Input -> tải {hf_repo} ({hf_split}) từ Hugging Face")
        return load_dataset(hf_repo, split=hf_split).to_pandas()
    except Exception as e:
        log(f"[{name}] KHÔNG lấy được dữ liệu ({str(e)[:150]}). Thêm vào Input hoặc bật Internet.")
        return None

## 2. Bộ câu hỏi chung (giống hệt bộ BAF và bộ WildGuardMix + SQLi)

In [ ]:
DECISIONS  = ["BLOCK", "ESCALATE_HUMAN", "TRIGGER_AI", "ALLOW"]
TRACE_OPTS = ["audit_log", "internet_cti", "syntax", "none"]
QUESTIONS = {
    "decision": {"type": "choice",
                 "instructions": "First-pass security verdict for this request.",
                 "criteria": {
                     "BLOCK": "Clear malicious signals; reject now.",
                     "ESCALATE_HUMAN": "High risk or high impact; hold for admin review.",
                     "TRIGGER_AI": "Suspicious but inconclusive; investigate further.",
                     "ALLOW": "No meaningful risk; let it through."}},
    "trace_group": {"type": "choice",
                    "instructions": "If investigated, which agent's evidence matters most?",
                    "criteria": {
                        "audit_log": "History, velocity and identity records in logs/DB.",
                        "internet_cti": "Device, network, session and geo signals.",
                        "syntax": "Structure of code, SQL or prompt payload.",
                        "none": "No investigation lead."}},
    "needs_investigation": {"type": "noul",
                            "instructions": "Is the request alone insufficient for a confident verdict?",
                            "criteria": {"false": "The request alone is enough to decide.",
                                         "true": "More evidence is needed before deciding."}},
    "risk": {"type": "score", "instructions": "How risky is this request?",
             "criteria": ["none", "low", "elevated", "high"]},
    "urgency": {"type": "score", "instructions": "How urgently must security act?",
                "criteria": ["routine", "soon", "prompt", "immediate"]},
}
QUESTIONS_STR = jdump(QUESTIONS)

## 3. Làm sạch, rút gọn, khoá chống trùng

In [ ]:
ZW     = re.compile("[\u200b\u200c\u200d\u2060\ufeff]")
CTRL   = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")
REPEAT = re.compile(r"(.{8,200}?)\1{3,}", re.S)

def clean_text(t):
    """Giữ dấu hiệu làm rối (zero-width -> [ZW]); nén đoạn lặp liên tiếp."""
    if not isinstance(t, str):
        return None
    t = CTRL.sub("", t)
    t = ZW.sub("[ZW]", t)
    t = re.sub(r"[ \t]+", " ", t)
    t = re.sub(r"\n{3,}", "\n\n", t).strip()
    t = REPEAT.sub(lambda m: f"{m.group(1)}[x{len(m.group(0)) // len(m.group(1))}] ", t)
    return t or None

def shorten(t, head=HEAD, tail=TAIL):
    """Giữ đầu + cuối: lệnh tấn công thường nằm cuối, bối cảnh nằm đầu."""
    if len(t) <= head + tail + 40:
        return t, False
    return f"{t[:head]} …[{len(t) - head - tail} chars cut]… {t[-tail:]}", True

def norm_key(t):
    """Khoá chống trùng chính xác (không phân biệt hoa/thường, dấu câu, khoảng trắng)."""
    t = unicodedata.normalize("NFKC", t).lower()
    return hashlib.md5(re.sub(r"\W+", " ", t).strip().encode()).hexdigest()

def sql_template(t):
    """Khoá 'khuôn' SQL: thay chuỗi và số bằng ký hiệu -> các biến thể của cùng 1 payload
    luôn rơi vào cùng 1 tập (chống rò rỉ gần-trùng)."""
    t = t.lower()
    t = re.sub(r"'[^']*'|\"[^\"]*\"", "S", t)
    t = re.sub(r"\b0x[0-9a-f]+\b|\b\d+(\.\d+)?\b", "N", t)
    t = re.sub(r"\s+", " ", t).strip()
    return hashlib.md5(t.encode()).hexdigest()

def hash_bucket(key):
    return int(key[:8], 16) % 10_000 / 10_000

def length_band(n):
    return "short" if n < 200 else "medium" if n < 1000 else "long" if n < 4000 else "very_long"

In [ ]:
def minhash_clusters(texts, n_perm=64, bands=8, k=5):
    """Gộp câu GẦN trùng bằng MinHash + LSH (8 band x 8 hàng -> ngưỡng Jaccard ~0.77).
    Gom theo 'câu đại diện' (không nối chuỗi) -> câu cùng khuôn nhưng KHÁC chủ đề không bị gộp."""
    P = (1 << 31) - 1
    rng = np.random.default_rng(SEED)
    a = rng.integers(1, P, n_perm, dtype=np.int64)
    b = rng.integers(0, P, n_perm, dtype=np.int64)
    rows = n_perm // bands
    buckets, cluster = {}, np.empty(len(texts), dtype=np.int64)
    for i, t in enumerate(texts):
        w = re.findall(r"\w+", t.lower())
        sh = {" ".join(w[j:j + k]) for j in range(max(1, len(w) - k + 1))}
        h = np.array([zlib.crc32(s_.encode()) % P for s_ in sh], dtype=np.int64)
        sig = ((a[:, None] * h[None, :] + b[:, None]) % P).min(1)
        keys = [(band, sig[band * rows:(band + 1) * rows].tobytes()) for band in range(bands)]
        rep = next((buckets[k_] for k_ in keys if k_ in buckets), i)   # đại diện đã gặp, hoặc chính nó
        cluster[i] = rep
        if rep == i:
            for k_ in keys:
                buckets.setdefault(k_, i)
    return cluster

## 4. Luật cú pháp

In [ ]:
PROMPT_RULES = [
    (1.0, r"\b(ignore|disregard|forget)\b.{0,30}\b(previous|prior|above|all|your)\b.{0,20}\b(instruction|prompt|rule|direction)s?"),
    (1.0, r"\b(system prompt|initial prompt|hidden instructions?)\b"),
    (0.8, r"\b(developer mode|DAN|jailbreak|unfiltered|no (restrictions|limits|rules))\b"),
    (0.5, r"\b(you are now|act as|pretend (to be|you are)|roleplay|role-play|from now on)\b"),
    (0.5, r"\b(hypothetical(ly)?|fictional (story|world|scenario)|for a (novel|story|screenplay))\b"),
    (0.6, r"(<\|im_start\|>|\[/?INST\]|</?system>|###\s*(system|instruction))"),
    (0.6, r"[A-Za-z0-9+/]{40,}={0,2}"),
    (0.4, r"\[ZW\]"),
]
SQL_RULES = [
    (1.0, r"\b(or|and)\s+(['\"]?)(\w+)\2\s*(=|like)\s*\2\3\2"),   # tautology: OR 1=1, OR 'a'='a'
    (1.0, r"'\s*(or|and)\s*'[^']*'\s*=\s*'"),                     # ' or ''='
    (1.0, r"'\s*\)?\s*(--|#|/\*)"),                               # đóng chuỗi rồi comment: admin' --
    (1.0, r"\bunion\b(\s+all)?\s+select\b"),
    (1.0, r";\s*(drop|delete|insert|update|exec|shutdown|truncate)\b"),
    (1.0, r"\b(sleep|benchmark|pg_sleep)\s*\(|waitfor\s+delay"),
    (0.8, r"\binformation_schema\b|@@version|\bsysobjects\b|\bload_file\s*\(|\bxp_cmdshell\b"),
    (0.5, r"(--|#|/\*)"),
    (0.5, r"\bchar\s*\(|0x[0-9a-f]{6,}|%27|%22|%3d"),
]
P_RX = [(w, re.compile(p, re.I)) for w, p in PROMPT_RULES]
S_RX = [(w, re.compile(p, re.I)) for w, p in SQL_RULES]
HIGH_IMPACT = re.compile(r"^\s*(drop|truncate|alter|grant|revoke)\b", re.I)
WRITE_OP    = re.compile(r"^\s*(delete|update)\b", re.I)

def sql_impact(sql):
    """Thao tác cần người duyệt: đổi cấu trúc / phân quyền (DROP, TRUNCATE, ALTER, GRANT, REVOKE),
    hoặc DELETE/UPDATE KHÔNG có WHERE (tác động toàn bảng).
    DELETE/UPDATE có WHERE là nghiệp vụ thường ngày -> không tính."""
    return bool(HIGH_IMPACT.match(sql)) or (bool(WRITE_OP.match(sql))
                                            and not re.search(r"\bwhere\b", sql, re.I))

def suspicion(text, kind):
    rules = S_RX if kind == "sql" else P_RX
    return min(3.0, sum(w for w, rx in rules if rx.search(text)))

# vỏ bọc "hợp thức hoá" hay gặp trong FinRED / jailbreak tài chính
PRETEXT = re.compile(r"\b(red[- ]?team|penetration test|pen[- ]?test|security (training|exercise|research|audit|assessment)"
                     r"|authori[sz]ed (engagement|test|exercise)|simulation|educational purposes?|for (research|training) purposes"
                     r"|hypothetical(ly)?|role[- ]?play|pretend|imagine you are|you are (now )?(a|an) )", re.I)

## 5. Đọc từng nguồn → bản ghi chuẩn

In [ ]:
def rec(text, source, split, p_mal, agree, adversarial=False, impact=False,
        hard_neg=False, label_raw=None, sub=None, group=None):
    return dict(text=text, kind="prompt", source=source, split=split, p_mal=float(p_mal),
                agree=float(agree), adversarial=bool(adversarial), impact=bool(impact),
                hard_neg=bool(hard_neg), context=None, label_raw=label_raw, subcategory=sub,
                group=group)

FINRED_FILE = re.compile(r"^(R\d+_\d+).*\.(csv|parquet|jsonl?)$", re.I)   # vd R1_1_prompts.csv
PROMPT_COLS = ["prompt", "prompts", "text", "question", "input", "instruction"]

def read_finred():
    """Đọc FinRED dạng nhiều file theo loại đe dọa (R1_1_prompts.csv, ...) hoặc một file gộp.
    Mã loại đe dọa lấy từ cột category_prefix nếu có, nếu không thì lấy từ TÊN FILE."""
    files = sorted(set(find_files("finred")) | {
        f for f in glob.glob(f"{INPUT_DIR}/**/*", recursive=True)
        if os.path.isfile(f) and FINRED_FILE.match(os.path.basename(f))})
    if files:
        log(f"[FinRED] đọc {len(files)} file từ Input: {[os.path.basename(f) for f in files][:6]}"
            + (" ..." if len(files) > 6 else ""))
        parts = []
        for f in files:
            try:
                d = read_any(f)
            except Exception as e:
                log(f"[FinRED] bỏ qua {os.path.basename(f)}: {e}"); continue
            d.columns = [str(c).strip().lower() for c in d.columns]
            pcol = next((c for c in PROMPT_COLS if c in d.columns), None)
            if pcol is None:
                log(f"[FinRED] bỏ qua {os.path.basename(f)}: không thấy cột prompt (có: {list(d.columns)})"); continue
            m = FINRED_FILE.match(os.path.basename(f))
            cat = d["category_prefix"] if "category_prefix" in d.columns else (m.group(1) if m else None)
            if cat is None:
                log(f"[FinRED] bỏ qua {os.path.basename(f)}: không xác định được loại đe dọa"); continue
            parts.append(pd.DataFrame({"prompt": d[pcol], "category_prefix": cat}))
        df = pd.concat(parts, ignore_index=True) if parts else None
    else:
        df = load_source("FinRED", ["finred"], "datumo/FinRED", "test")
        if df is not None:
            df.columns = [str(c).strip().lower() for c in df.columns]
    if df is None or not {"prompt", "category_prefix"} <= set(df.columns):
        log("[FinRED] KHÔNG đọc được dữ liệu. Thêm các file R*_prompts.csv vào Input hoặc bật Internet.")
        return []
    out = []
    for r in df.dropna(subset=["prompt"]).to_dict("records"):
        cat = str(r["category_prefix"]).strip().upper()
        top = cat.split("_")[0]
        if top not in FINRED_RISK or len(str(r["prompt"]).strip()) < 10:
            continue
        out.append(rec(str(r["prompt"]), "finred", None, FINRED_RISK[top], 0.9,
                       adversarial=bool(PRETEXT.search(str(r["prompt"]))),
                       impact=top in FINRED_IMPACT, label_raw=top, sub=cat))
    log(f"[FinRED] {len(out):,} prompt độc hại; theo nhóm: "
        f"{pd.Series([o['label_raw'] for o in out]).value_counts().sort_index().to_dict()}; "
        f"{len(set(o['subcategory'] for o in out))} loại chi tiết")
    return out

def sample_length_matched(text, ref_lengths, n, name):
    """Lấy n câu sao cho phân bố độ dài giống ref_lengths (10 khoảng theo phân vị).
    Khoảng nào thiếu câu thì lấy hết những gì có - KHÔNG bù bằng câu lạc đề."""
    edges = np.unique(np.quantile(ref_lengths, np.linspace(0, 1, 11)))
    edges[0], edges[-1] = 0, np.inf
    bins = pd.cut(text.str.len(), edges, labels=False, include_lowest=True)
    want = (pd.cut(pd.Series(ref_lengths), edges, labels=False, include_lowest=True)
            .value_counts(normalize=True) * n).round().astype(int)
    got = pd.concat([text[bins == bi].sample(n=min(k, int((bins == bi).sum())), random_state=SEED)
                     for bi, k in want.items()])
    log(f"[{name}] lấy {len(got):,}/{n:,} câu khớp độ dài: trung vị {int(got.str.len().median())} ký tự "
        f"(FinRED: {int(np.median(ref_lengths))})")
    return got

def read_finance(ref_lengths=None):
    df = load_source("finance-alpaca", ["finance-alpaca", "finance_alpaca"], "gbharti/finance-alpaca", "train")
    if df is None: return []
    if "instruction" not in df.columns:
        log(f"[finance-alpaca] thiếu cột instruction (có: {list(df.columns)})"); return []
    inp = df["input"].fillna("") if "input" in df else ""
    text = (df["instruction"].fillna("").str.strip() + "\n" + inp.astype(str).str.strip()).str.strip()
    text = text[text.str.len() >= 10].drop_duplicates()
    # finance-alpaca chủ yếu là câu NGẮN -> lấy ngẫu nhiên; phần câu dài do Sujet đảm nhận
    text = text.sample(n=min(FINANCE_MAX, len(text)), random_state=SEED)
    log(f"[finance-alpaca] {len(text):,} câu hỏi tài chính vô hại")
    return [rec(t, "finance_alpaca", None, 0.0, 0.95, label_raw="benign") for t in text]

def read_sujet(ref_lengths):
    df = load_source("Sujet-Finance", ["sujet"], "sujet-ai/Sujet-Finance-Instruct-177k", "train")
    if df is None: return []
    col = "user_prompt" if "user_prompt" in df.columns else ("inputs" if "inputs" in df.columns else None)
    if col is None:
        log(f"[Sujet-Finance] thiếu cột user_prompt (có: {list(df.columns)})"); return []
    text = df[col].dropna().astype(str).str.strip()
    ttype = df.loc[text.index, "task_type"].astype(str) if "task_type" in df.columns else pd.Series("unknown", index=text.index)
    lo, hi = np.quantile(ref_lengths, 0.01) * 0.8, np.quantile(ref_lengths, 0.99) * 1.2
    keep = text.str.len().between(lo, hi)
    text = text[keep].drop_duplicates()
    log(f"[Sujet-Finance] {len(text):,} câu trong khoảng độ dài {int(lo)}-{int(hi)} ký tự")
    if not len(text): return []
    if LENGTH_MATCH:
        text = sample_length_matched(text, ref_lengths, SUJET_MAX, "Sujet-Finance")
    else:
        text = text.sample(n=min(SUJET_MAX, len(text)), random_state=SEED)
    out = [rec(t, "sujet_finance", None, 0.0, 0.95, label_raw="benign_long", sub=ttype.get(i, "unknown"))
           for i, t in text.items()]
    log(f"[Sujet-Finance] loại tác vụ: {pd.Series([o['subcategory'] for o in out]).value_counts().to_dict()}")
    return out

REFUSAL = re.compile(r"^\s*(I (cannot|can't|won't|am unable to|must decline to) (provide|help|assist|create|write|give))", re.I)
OFFENSIVE = re.compile(r"\b(design|enhance|automate|develop|build|create)\b.{0,50}\b(bootkit|rootkit|persistence|c2|"
                       r"command[- ]and[- ]control|obfuscation|hiding|evasion|exploit(ation)?|malware|ransomware|keylogger)\b", re.I)

def read_trendyol():
    df = load_source("Trendyol", ["trendyol"],
                     "Trendyol/Trendyol-Cybersecurity-Instruction-Tuning-Dataset", "train")
    if df is None: return []
    if "user" not in df.columns:
        log(f"[Trendyol] thiếu cột user (có: {list(df.columns)})"); return []
    df = df.dropna(subset=["user"])
    df = df[df["user"].str.len() >= 10].reset_index(drop=True)
    log(f"[Trendyol] gộp câu gần-trùng trên {len(df):,} câu (MinHash, ~1 phút)...")
    df["cluster"] = minhash_clusters(df["user"].tolist())
    df = df.drop_duplicates("cluster")
    log(f"[Trendyol] còn {len(df):,} cụm khác nhau")
    df = df.sample(n=min(TRENDYOL_MAX, len(df)), random_state=SEED)
    out = []
    for r in df.to_dict("records"):
        refused = bool(REFUSAL.match(str(r.get("assistant", ""))))
        offensive = bool(OFFENSIVE.search(r["user"]))
        if refused:     p, ag, lab = 0.8, 0.7, "refused"         # người soạn dataset coi là độc hại
        elif offensive: p, ag, lab = 0.45, 0.7, "offensive_verb"  # vùng mơ hồ -> TRIGGER_AI
        else:           p, ag, lab = 0.0, 0.9, "defensive"        # câu hỏi phòng thủ hợp lệ
        out.append(rec(r["user"], "trendyol", None, p, ag, hard_neg=(lab == "defensive"),
                       label_raw=lab, group=f"tr{r['cluster']}"))
    log(f"[Trendyol] nhãn yếu: {pd.Series([o['label_raw'] for o in out]).value_counts().to_dict()}")
    return out

## 6. Đọc dữ liệu

In [ ]:
fin_rows = read_finred()
fin_len = [len(r["text"]) for r in fin_rows]
rows = fin_rows + read_finance(fin_len) + read_sujet(fin_len) + read_trendyol()
assert any(r["source"] == "finred" for r in rows), "Không đọc được FinRED - xem thông báo ở trên."
df = pd.DataFrame(rows)
df["orig_len"] = df["text"].map(lambda t: len(t) if isinstance(t, str) else 0)
df["text"] = df["text"].map(clean_text)
df = df.dropna(subset=["text"])
df = df[df["text"].str.len() >= 3].reset_index(drop=True)
df["key"] = df["text"].map(norm_key)
df["group"] = df["group"].fillna(df["key"])
pd.crosstab(df["source"], df["label_raw"])

## 7. Chia train / valid / test

- **FinRED – test_unseen**: mỗi nhóm R1–R5 giữ lại **trọn một loại chi tiết** (vd R2_5) → model không bao giờ thấy loại này khi học.
  Đây là bài kiểm tra quan trọng nhất: model có nhận ra **kiểu tấn công tài chính mới** không?
- **FinRED – phần còn lại**: tách theo hash (25% test, 10% valid) → test ngẫu nhiên đo khả năng nhận lại kiểu tấn công **đã học**.
- **finance-alpaca, Trendyol**: 80/10/10 theo hash; Trendyol chia theo **cụm gần-trùng** → các biến thể của cùng một câu hỏi luôn chung tập.
- Chống trùng xuyên nguồn, xuyên tập: giữ bản ở tập ưu tiên cao hơn (test > valid > train).

In [ ]:
b = df["group"].map(lambda g: hash_bucket(hashlib.md5(str(g).encode()).hexdigest()))
fin = df["source"] == "finred"
test_p  = np.where(fin, FINRED_TEST_PCT, TEST_PCT)
valid_p = np.where(fin, FINRED_VALID_PCT, VALID_PCT)
df["split"] = np.where(b < test_p, "test", np.where(b < test_p + valid_p, "valid", "train"))

# --- test_unseen: mỗi nhóm R giữ lại trọn 1 loại chi tiết, KHÔNG xuất hiện ở train/valid/test ---
subs = sorted(df.loc[fin, "subcategory"].unique())
if HOLDOUT_SUBCATS == "auto":
    rng_h = np.random.default_rng(SEED)
    HOLDOUT = []
    for grp in sorted(FINRED_RISK):
        cands = [c for c in subs if c.split("_")[0] == grp]
        if len(cands) >= 2:                      # chỉ giữ lại khi nhóm còn ít nhất 1 loại để học
            HOLDOUT.append(str(rng_h.choice(cands)))
else:
    HOLDOUT = [c.upper() for c in HOLDOUT_SUBCATS if c.upper() in subs]
df.loc[fin & df["subcategory"].isin(HOLDOUT), "split"] = "test_unseen"
log(f"Loại đe dọa giữ lại cho test_unseen: {HOLDOUT} "
    f"({int((df.split == 'test_unseen').sum()):,} prompt = {(df.split == 'test_unseen')[fin].mean():.0%} FinRED)")

df["_p"] = df["split"].map({"test_unseen": 0, "test": 1, "valid": 2, "train": 3})
n0 = len(df)
df = df[df["_p"] == df.groupby("key")["_p"].transform("min")]
g = df.groupby("key")["p_mal"]
conflict = g.transform("std").fillna(0) > 0.3
df["p_mal"] = g.transform("mean")
df.loc[conflict, "agree"] = 0.6
df = df.drop_duplicates("key").drop(columns="_p").reset_index(drop=True)
log(f"Chống trùng: {n0:,} -> {len(df):,} dòng; {int(conflict.sum())} dòng nhãn mâu thuẫn")
df["susp"] = [suspicion(t, "prompt") for t in df["text"]]
pd.crosstab(df["split"], df["source"], margins=True)

## 8. Gold mềm (cùng hàm với bộ WildGuardMix + SQLi)

In [ ]:
def soft_labels(d, k=N_ANNOTATORS, noise=NOISE, protect_legit=PROTECT_LEGIT):
    rng = np.random.default_rng(SEED)
    n = len(d)
    p   = d["p_mal"].values[:, None]
    ag  = d["agree"].values[:, None]
    sus = d["susp"].values[:, None]
    adv = d["adversarial"].values[:, None].astype(float)
    hard = d["hard_neg"].values[:, None].astype(float)
    sql = (d["kind"].values == "sql")[:, None]
    impact = d["impact"].values[:, None]

    flip = rng.random((n, k)) > ag                       # annotator không tin nhãn gốc
    pv = np.where(flip, 1 - p, p)
    latent = (rng.uniform(2.6, 3.2, k) * pv + rng.uniform(.3, .6, k) * sus
              + 0.6 * hard + 0.3 * adv + rng.normal(0, noise, (n, k)))
    t1, t2, t3 = rng.uniform(.5, .8, k), rng.uniform(1.3, 1.7, k), rng.uniform(2.1, 2.5, k)
    lvl = (latent >= t1).astype(int) + (latent >= t2) + (latent >= t3)
    # ca "sát ranh giới": điểm rủi ro gần một ngưỡng -> quyết định dễ đổi nếu có thêm bằng chứng
    margin = np.minimum(np.minimum(abs(latent - t1), abs(latent - t2)), abs(latent - t3))
    ev = sus + 1.8 * pv + 0.5 * adv + rng.normal(0, .2, (n, k))   # độ rõ của bằng chứng
    e_weak, e_strong = rng.uniform(.4, .8, k), rng.uniform(1.2, 1.8, k)
    imp = impact & (rng.random((n, k)) < .9)

    # chỉ số theo DECISIONS: 0 BLOCK, 1 ESCALATE_HUMAN, 2 TRIGGER_AI, 3 ALLOW
    dec = np.select([imp & (lvl <= 2), lvl == 0, (lvl == 1) & (ev < e_weak), lvl == 1,
                     lvl == 2, (lvl == 3) & (ev >= e_strong)],
                    [1, 3, 3, 2, 2, 0], default=1)
    if protect_legit:                                    # mẫu gốc vô hại: tối đa ESCALATE
        dec = np.where((dec == 0) & (p < 0.5), 1, dec)
    # cần điều tra: (có rủi ro nhưng bằng chứng chưa đủ mạnh) HOẶC (ca sát ranh giới)
    need = ((lvl >= 1) & (ev < e_strong)) | (margin < 0.3)

    # syntax: có cấu trúc tấn công (luật khớp, vỏ jailbreak, hoặc SQL độc hại)
    tscore = sus + 1.0 * adv + 0.5 * pv + 1.0 * (sql & (pv >= .5)) + rng.normal(0, .2, (n, k))
    trace = np.where(tscore >= rng.uniform(.6, .9, k), 2, 3)
    urg = np.clip(np.where(imp, np.maximum(lvl, 1), lvl) + ((lvl == 3) & (ev >= e_strong)), 0, 3)

    frac = lambda m, c: np.stack([(m == i).mean(1) for i in range(c)], 1)
    return {"decision": frac(dec, 4), "trace_group": frac(trace, 4),
            "needs_investigation": need.mean(1),
            "risk": frac(np.maximum(lvl, imp.astype(int)), 4), "urgency": frac(urg, 4)}

def conf(p):
    kk = len(p)
    return float(round((max(p) - 1 / kk) / (1 - 1 / kk), 6))

def gold_row(i, sl):
    r6 = lambda x: float(round(float(x), 6))
    d, t, pt = sl["decision"][i], sl["trace_group"][i], float(sl["needs_investigation"][i])
    g = {"decision": {"type": "choice", "label": DECISIONS[int(d.argmax())], "confidence": conf(d),
                      "probabilities": {a: r6(b) for a, b in zip(DECISIONS, d)}},
         "trace_group": {"type": "choice", "label": TRACE_OPTS[int(t.argmax())], "confidence": conf(t),
                         "probabilities": {a: r6(b) for a, b in zip(TRACE_OPTS, t)}},
         "needs_investigation": {"type": "noul", "label": "true" if pt >= .5 else "false",
                                 "noul": r6(pt), "confidence": conf([pt, 1 - pt]),
                                 "probabilities": {"false": r6(1 - pt), "true": r6(pt)}}}
    for q in ["risk", "urgency"]:
        p = sl[q][i]
        g[q] = {"type": "score", "label": str(int(p.argmax())),
                "score": r6((p * np.arange(4)).sum()), "confidence": conf(p),
                "probabilities": {str(j): r6(v) for j, v in enumerate(p)}}
    return g

In [ ]:
SL = soft_labels(df)
# prompt FinRED nêu rõ ý đồ -> bản thân câu đã đủ để kết luận (kể cả khi ESCALATE vì tác hại gián tiếp)
SL["needs_investigation"] = np.where(df["source"].values == "finred",
                                     SL["needs_investigation"] * FINRED_NEED_SCALE, SL["needs_investigation"])
df["decision"] = np.array(DECISIONS)[SL["decision"].argmax(1)]
df["trace"] = np.array(TRACE_OPTS)[SL["trace_group"].argmax(1)]
pd.crosstab([df["source"], df["label_raw"]], df["decision"])

## 9. Dựng state và ghi file

In [ ]:
def build_state(r):
    txt, cut = shorten(r["text"])
    # mọi nguồn dùng CHUNG một kênh -> model không học được "kênh ngân hàng = tấn công"
    return {"workflow": "chat_request", "channel": "banking_chatbot",
            "request": {"message": txt, "length": length_band(r["orig_len"]), "truncated": cut}}

files = {s: open(f"{OUT_DIR}/{s}.jsonl", "w", encoding="utf-8")
         for s in ["train", "valid", "test", "test_unseen"]}
lens, cuts = [], 0
for i, r in enumerate(df.to_dict("records")):
    st = build_state(r)
    cuts += st["request"]["truncated"]
    st = jdump(st); lens.append(len(st))
    files[r["split"]].write(jdump({"state": st, "questions": QUESTIONS_STR,
                                   "gold": jdump(gold_row(i, SL))}) + "\n")
for f in files.values(): f.close()
df[["key", "group", "split", "source", "label_raw", "subcategory", "adversarial", "p_mal",
    "agree", "susp", "impact", "decision", "trace"]].rename(columns={"key": "group_key", "group": "cluster_key"}) \
    .to_csv(f"{OUT_DIR}/finred_meta.csv", index=False)
log(f"Đã ghi {len(df):,} dòng; state TB {np.mean(lens):.0f} ký tự, max {max(lens)}; {cuts} dòng bị rút gọn")

## 10. Kiểm tra chất lượng

In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

P, need = SL["decision"], SL["needs_investigation"]
fin = df.source == "finred"; fa = df.source.isin(["finance_alpaca", "sujet_finance"]); tr = df.source == "trendyol"
ov = lambda a, b, col: len(set(df.loc[df.split == a, col]) & set(df.loc[df.split == b, col]))

# Đường tắt: chỉ nhìn ĐỘ DÀI hoặc TỪ VỰNG mà đã phân biệt được FinRED với câu vô hại?
y = fin.astype(int)
te = df.split == "test"; trn = df.split == "train"
len_auc = roc_auc_score(y[te], df.loc[te, "orig_len"]) if y[te].nunique() > 1 else float("nan")
vec = TfidfVectorizer(min_df=2, max_features=50_000, ngram_range=(1, 2))
clf = LogisticRegression(max_iter=1000).fit(vec.fit_transform(df.loc[trn, "text"]), y[trn])
tfidf_auc = roc_auc_score(y[te], clf.predict_proba(vec.transform(df.loc[te, "text"]))[:, 1]) \
    if y[te].nunique() > 1 else float("nan")

baf_q = next(iter(glob.glob(f"{INPUT_DIR}/**/rexbot_baf/train.jsonl", recursive=True)), None)
checks = {
    "Câu tài chính vô hại (finance-alpaca + Sujet) bị BLOCK (phải = 0)": int(((df.decision == "BLOCK") & fa).sum()),
    "Tỉ lệ ALLOW ở câu tài chính vô hại (nên >= 90%)": round(float((df.decision == "ALLOW")[fa].mean()), 3) if fa.any() else "n/a",
    "Tỉ lệ BLOCK+ESCALATE ở FinRED (nên >= 85%)": round(float(df.decision[fin].isin(["BLOCK", "ESCALATE_HUMAN"]).mean()), 3),
    "Tỉ lệ ALLOW ở FinRED (nên <= 5%)": round(float((df.decision == "ALLOW")[fin].mean()), 3),
    "Trendyol 'defensive' được ALLOW (nên >= 80%)":
        round(float((df.decision == "ALLOW")[tr & (df.label_raw == "defensive")].mean()), 3) if tr.any() else "n/a",
    "Văn bản trùng train-test / train-valid / valid-test (phải = 0)":
        (ov("train", "test", "key"), ov("train", "valid", "key"), ov("valid", "test", "key")),
    "Cụm gần-trùng (Trendyol) trùng train-test (phải = 0)": ov("train", "test", "group"),
    "Loại giữ lại (test_unseen) xuất hiện ở train/valid/test (phải = 0)":
        int((fin & df.subcategory.isin(HOLDOUT) & (df.split != "test_unseen")).sum()),
    "Số nhóm R có 1 loại giữ lại (nên = 5)": len({h.split("_")[0] for h in HOLDOUT}),
    "Đủ 5 nhóm R1-R5 ở cả train/valid/test":
        all(set(df.loc[fin & (df.split == s), "label_raw"]) == set(FINRED_RISK) for s in ["train", "valid", "test"]),
    "Tương quan needs_investigation với P(ESC)+P(TRIGGER) (phải < 0.95)":
        round(float(np.corrcoef(need, P[:, 1] + P[:, 2])[0, 1]), 3),
    "AUC chỉ dùng ĐỘ DÀI để tách FinRED / vô hại (cao = có đường tắt, nên < 0.85)": round(float(len_auc), 3),
    "AUC TF-IDF + LogReg trên test (gần 1.0 = bài toán dễ theo từ vựng)": round(float(tfidf_auc), 3),
}
if baf_q:
    checks["questions giống hệt bộ BAF"] = json.loads(open(baf_q).readline())["questions"] == QUESTIONS_STR

# ---- So sánh test ngẫu nhiên và test_unseen bằng 1 model đơn giản (TF-IDF + LogReg) ----
# Cùng một tập câu vô hại (test) làm mẫu âm; chỉ khác phần FinRED: loại ĐÃ học vs loại CHƯA thấy.
benign_te = te & ~fin
def eval_set(pos_mask, name):
    m = (pos_mask | benign_te).values
    yy = fin[m].astype(int).values
    pp = clf.predict_proba(vec.transform(df.loc[m, "text"]))[:, 1]
    pos = yy == 1
    return {"tập": name, "số FinRED": int(pos.sum()), "số vô hại": int((~pos).sum()),
            "AUC TF-IDF": round(roc_auc_score(yy, pp), 3) if 0 < pos.sum() < len(yy) else float("nan"),
            "Recall FinRED @0.5": round(float((pp[pos] >= 0.5).mean()), 3) if pos.any() else float("nan"),
            "AUC độ dài": round(roc_auc_score(yy, df.loc[m, "orig_len"]), 3) if 0 < pos.sum() < len(yy) else float("nan")}
cmp_tbl = pd.DataFrame([eval_set(te & fin, "test (loại đã học)"),
                        eval_set((df.split == "test_unseen") & fin, "test_unseen (loại chưa thấy)")]).set_index("tập")
unseen_detail = pd.crosstab(df.loc[df.split == "test_unseen", "subcategory"],
                            df.loc[df.split == "test_unseen", "decision"])

report = "\n".join([
    "=== REX BOT / FinRED REPORT ===", "",
    f"Loại đe dọa giữ lại cho test_unseen: {HOLDOUT}", "",
    "Số dòng theo tập x nguồn:", pd.crosstab(df["split"], df["source"], margins=True).to_string(), "",
    "Tỉ lệ nhãn decision theo tập (%):",
    (pd.crosstab(df["split"], df["decision"], normalize="index") * 100).round(1).to_string(), "",
    "Decision theo nguồn và nhãn gốc:",
    pd.crosstab([df["source"], df["label_raw"]], df["decision"]).to_string(), "",
    "FinRED: decision theo loại đe dọa chi tiết (%):",
    (pd.crosstab(df.loc[fin, "subcategory"], df.loc[fin, "decision"], normalize="index") * 100).round(0).to_string(), "",
    "trace_group theo nguồn (%):",
    (pd.crosstab(df["source"], df["trace"], normalize="index") * 100).round(1).to_string(), "",
    "SO SÁNH TEST NGẪU NHIÊN vs TEST_UNSEEN (baseline TF-IDF + LogReg, chỉ để tham khảo):",
    cmp_tbl.to_string(),
    "  -> Khoảng cách lớn giữa 2 dòng = model nhận ra VĂN PHONG đã học hơn là Ý ĐỒ tấn công.", "",
    "Gold của test_unseen theo loại:", unseen_detail.to_string(), "",
    "KIỂM TRA TỰ ĐỘNG:", *[f"  - {k}: {v}" for k, v in checks.items()], "",
    f"Độ dài state: TB {np.mean(lens):.0f}, max {max(lens)} ký tự; {cuts} dòng bị rút gọn",
])
open(f"{OUT_DIR}/report.txt", "w", encoding="utf-8").write(report)
print(report)

In [ ]:
# Xem 1 mẫu mỗi nguồn
for src_name in ["finred", "finance_alpaca", "sujet_finance", "trendyol"]:
    idx = df.index[df.source == src_name]
    if len(idx) == 0: continue
    i = idx[0]; g = gold_row(i, SL)
    print(f"=== {src_name} | {df.at[i, 'label_raw']} ===\n{df.at[i, 'text'][:350]}")
    print("decision:", g["decision"]["probabilities"], "| trace:", g["trace_group"]["label"], "\n")

## 11. Đóng gói

In [ ]:
zip_path = OUT_DIR.rstrip("/") + ".zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir(OUT_DIR)):
        z.write(os.path.join(OUT_DIR, f), arcname=f"rexbot_finred/{f}")
for f in sorted(os.listdir(OUT_DIR)):
    print(f"{f:18s} {os.path.getsize(os.path.join(OUT_DIR, f)) / 1e6:8.1f} MB")
print(f"\nFile nén: {zip_path} ({os.path.getsize(zip_path) / 1e6:.1f} MB)")

## Ghi chú
- **Giấy phép:** FinRED là CC BY-NC 4.0 (chỉ phi thương mại). Dataset upload lên Kaggle và bộ đã gộp nên để **Private**.
- **Đường tắt độ dài:** prompt FinRED dài 440-960 ký tự, còn finance-alpaca và Trendyol phần lớn ngắn. Sujet-Finance
  được lấy mẫu khớp đúng phân bố độ dài FinRED để model không thể đoán nhãn chỉ bằng độ dài.
- **Gộp:** cùng 5 câu hỏi và format với bộ BAF và bộ WildGuardMix + SQLi → nối các file `train/valid/test.jsonl`.
- **`test_unseen.jsonl` KHÔNG được gộp vào train/valid.** Dùng riêng sau khi train để báo cáo khả năng nhận ra
  kiểu tấn công tài chính mới. Muốn so sánh công bằng, đánh giá model trên `test_unseen.jsonl` cùng với phần câu vô hại
  của `test.jsonl` (giống bảng so sánh trong report).